# Medical Abstracts TC Corpus — dataset preparation

This notebook prepares the Medical Abstracts TC Corpus for modeling. The main
objectives are to resolve label ambiguity, remove duplicated texts, prevent data
leakage, and create reproducible stratified training, validation, and test sets.

The raw files are preserved without modification. All transformations performed
in this notebook will produce new files under `data/processed/`.

## 1. Environment setup

In [3]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

In [4]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = RAW_DATA_DIR / "medical_tc_train.csv"
TEST_PATH = RAW_DATA_DIR / "medical_tc_test.csv"
LABELS_PATH = RAW_DATA_DIR / "medical_tc_labels.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)
print("Train file found:", TRAIN_PATH.exists())

Project root: /home/mkiku/Medical-Triage
Raw data directory: /home/mkiku/Medical-Triage/data/raw
Train file found: True


## 2. Loading the raw data

The original training and test files are loaded and temporarily combined because
the exploratory analysis identified overlapping texts and conflicting labels
across the original split.

In [5]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
labels_df = pd.read_csv(LABELS_PATH)

raw_df = pd.concat(
    [train_raw, test_raw],
    ignore_index=True,
)

print("Original training shape:", train_raw.shape)
print("Original test shape:", test_raw.shape)
print("Combined shape:", raw_df.shape)

Original training shape: (11550, 2)
Original test shape: (2888, 2)
Combined shape: (14438, 2)


In [6]:
TEXT_COL = "medical_abstract"
TARGET_COL = "condition_label"

required_columns = {TEXT_COL, TARGET_COL}
missing_columns = required_columns.difference(raw_df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

if raw_df[[TEXT_COL, TARGET_COL]].isna().any().any():
    raise ValueError("The dataset contains missing text or target values.")

print("Initial validation completed successfully.")

Initial validation completed successfully.


## 3. Minimal text normalization

Only whitespace is normalized at this stage. Linguistic transformations such as
lowercasing, stopword removal, stemming, or lemmatization are intentionally
deferred to the modeling pipeline to prevent preprocessing decisions from
leaking across dataset splits.

In [7]:
prepared_df = raw_df.copy()

prepared_df[TEXT_COL] = (
    prepared_df[TEXT_COL]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

empty_text_count = prepared_df[TEXT_COL].eq("").sum()

print("Empty texts after normalization:", empty_text_count)

Empty texts after normalization: 0


In [8]:
if empty_text_count > 0:
    raise ValueError("Empty texts were found after whitespace normalization.")

## 4. Identifying ambiguous texts

A text is considered ambiguous when it is associated with more than one target
label. Because this project uses a single-label classifier, these texts cannot
be assigned a reliable class without introducing an arbitrary labeling rule.

Rather than selecting one label without clinical justification, all ambiguous
texts are excluded from the modeling dataset.

In [9]:
labels_per_text = (
    prepared_df
    .groupby(TEXT_COL)[TARGET_COL]
    .nunique()
)

ambiguous_texts = labels_per_text[
    labels_per_text > 1
].index

print("Unique texts:", len(labels_per_text))
print("Ambiguous unique texts:", len(ambiguous_texts))
print("Maximum labels per text:", labels_per_text.max())

Unique texts: 11227
Ambiguous unique texts: 2929
Maximum labels per text: 4


## 5. Removing ambiguous and duplicated texts

All occurrences of ambiguous texts are removed. For the remaining records, only
one occurrence of each medical abstract is retained. This produces a
single-label dataset in which every text appears exactly once.

In [10]:
clean_df = (
    prepared_df[
        ~prepared_df[TEXT_COL].isin(ambiguous_texts)
    ]
    .drop_duplicates(subset=[TEXT_COL])
    .reset_index(drop=True)
)

print("Rows before cleaning:", len(prepared_df))
print("Rows after cleaning:", len(clean_df))
print("Rows removed:", len(prepared_df) - len(clean_df))

Rows before cleaning: 14438
Rows after cleaning: 8298
Rows removed: 6140


In [11]:
assert not clean_df[TEXT_COL].duplicated().any()
assert clean_df.groupby(TEXT_COL)[TARGET_COL].nunique().max() == 1
assert clean_df[[TEXT_COL, TARGET_COL]].notna().all().all()

print("Clean dataset validation completed successfully.")

Clean dataset validation completed successfully.


## 6. Class distribution after cleaning

The class distribution is examined again after removing ambiguous and duplicated
texts. This verifies whether the cleaning procedure disproportionately affected
any target category.

In [12]:
class_distribution = (
    clean_df[TARGET_COL]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

class_distribution["percentage"] = (
    class_distribution["count"]
    / len(clean_df)
    * 100
)

class_distribution = (
    class_distribution
    .reset_index()
    .merge(
        labels_df,
        on=TARGET_COL,
        how="left",
    )
)

class_distribution

,condition_label,count,percentage,condition_name
0,1,2195,26.452157,neoplasms
1,2,699,8.423717,digestive system diseases
2,3,1049,12.641600,nervous system diseases
3,4,1961,23.632201,cardiovascular diseases
4,5,2394,28.850325,general pathological conditions


### Finding

The cleaned dataset remains imbalanced. Class 2 is the least represented,
whereas class 5 is the most frequent. A stratified split is therefore required
to preserve approximately the same class proportions across all datasets.
Macro F1-score will be used during modeling to give equal importance to each
class.

## 7. Stratified training, validation, and test split

The original split cannot be preserved because overlapping and conflicting texts
were found across its training and test sets. A new split is created from the
clean dataset.

Stratification preserves the target distribution, while the fixed random state
makes the result reproducible. The validation set will be used for model
selection, and the test set will remain isolated until the final evaluation.

In [13]:
model_train_df, temporary_df = train_test_split(
    clean_df,
    test_size=0.30,
    stratify=clean_df[TARGET_COL],
    random_state=RANDOM_STATE,
)

In [14]:
validation_df, model_test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    stratify=temporary_df[TARGET_COL],
    random_state=RANDOM_STATE,
)

In [15]:
model_train_df = model_train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
model_test_df = model_test_df.reset_index(drop=True)

print("Training shape:", model_train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", model_test_df.shape)

Training shape: (5808, 2)
Validation shape: (1245, 2)
Test shape: (1245, 2)


In [16]:
train_texts = set(model_train_df[TEXT_COL])
validation_texts = set(validation_df[TEXT_COL])
test_texts = set(model_test_df[TEXT_COL])

assert train_texts.isdisjoint(validation_texts)
assert train_texts.isdisjoint(test_texts)
assert validation_texts.isdisjoint(test_texts)

assert len(model_train_df) + len(validation_df) + len(model_test_df) == len(
    clean_df
)

print("No text overlap was found across the new splits.")

No text overlap was found across the new splits.


In [17]:
split_distribution = pd.concat(
    {
        "full": clean_df[TARGET_COL].value_counts(normalize=True),
        "train": model_train_df[TARGET_COL].value_counts(normalize=True),
        "validation": validation_df[TARGET_COL].value_counts(normalize=True),
        "test": model_test_df[TARGET_COL].value_counts(normalize=True),
    },
    axis=1,
).sort_index()

(split_distribution * 100).round(2)

,full,train,validation,test
condition_label,,,,
1,26.45,26.45,26.51,26.43
2,8.42,8.42,8.43,8.43
3,12.64,12.64,12.61,12.69
4,23.63,23.64,23.61,23.61
5,28.85,28.86,28.84,28.84


## 8. Final validation

Before exporting the datasets, each split is validated for missing values,
duplicated texts, unexpected labels, and structural inconsistencies.


In [18]:
expected_labels = set(labels_df[TARGET_COL])

datasets = {
    "train": model_train_df,
    "validation": validation_df,
    "test": model_test_df,
}

for dataset_name, dataset in datasets.items():
    assert list(dataset.columns) == [TARGET_COL, TEXT_COL]
    assert dataset[[TARGET_COL, TEXT_COL]].notna().all().all()
    assert not dataset[TEXT_COL].duplicated().any()
    assert set(dataset[TARGET_COL]).issubset(expected_labels)

    print(
        f"{dataset_name}: "
        f"{len(dataset)} rows, "
        f"{dataset[TARGET_COL].nunique()} classes"
    )

train: 5808 rows, 5 classes
validation: 1245 rows, 5 classes
test: 1245 rows, 5 classes


## 9. Exporting the processed datasets

The validated splits are saved under `data/processed/`. The raw files remain
unchanged, ensuring that the preparation process can be reproduced from the
original data.

In [19]:
PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

output_paths = {
    "train": PROCESSED_DATA_DIR / "train.csv",
    "validation": PROCESSED_DATA_DIR / "validation.csv",
    "test": PROCESSED_DATA_DIR / "test.csv",
    "labels": PROCESSED_DATA_DIR / "labels.csv",
}

model_train_df.to_csv(
    output_paths["train"],
    index=False,
)

validation_df.to_csv(
    output_paths["validation"],
    index=False,
)

model_test_df.to_csv(
    output_paths["test"],
    index=False,
)

labels_df.to_csv(
    output_paths["labels"],
    index=False,
)

In [20]:
for dataset_name, output_path in output_paths.items():
    print(
        f"{dataset_name}: "
        f"{output_path.relative_to(PROJECT_ROOT)} "
        f"({output_path.stat().st_size:,} bytes)"
    )

train: data/processed/train.csv (7,186,369 bytes)
validation: data/processed/validation.csv (1,558,997 bytes)
test: data/processed/test.csv (1,586,758 bytes)
labels: data/processed/labels.csv (157 bytes)


In [21]:
exported_train_df = pd.read_csv(output_paths["train"])
exported_validation_df = pd.read_csv(output_paths["validation"])
exported_test_df = pd.read_csv(output_paths["test"])
exported_labels_df = pd.read_csv(output_paths["labels"])

assert exported_train_df.equals(model_train_df)
assert exported_validation_df.equals(validation_df)
assert exported_test_df.equals(model_test_df)
assert exported_labels_df.equals(labels_df)

print("All processed files were exported and reloaded successfully.")

All processed files were exported and reloaded successfully.


## 10. Conclusions

The raw dataset contained duplicated abstracts, cross-split overlap, and texts
associated with multiple labels. Because the project adopts a single-label
classification formulation, ambiguous texts were removed rather than assigned
an arbitrary target.

After cleaning, 8,298 unique and unambiguous examples remained. They were split
into 5,808 training examples, 1,245 validation examples, and 1,245 test examples
using stratification and a fixed random state.

The resulting datasets:

- contain all five target classes;
- preserve approximately the same class distribution;
- contain no missing values;
- contain no duplicated texts;
- have no text overlap across splits;
- can be reproduced from the original raw files.

The processed data is now ready for baseline model development.